## Step 9: FL Simulation Loop

**Comparison type:** end-to-end. Each of the six methods keeps its *own* global model and its clients train on that model every round, so updates differ across methods (this is **not** a replay of fixed recorded updates). Methods are matched on seed, data split, client schedule (one shared participation stream) and attack definition; each method has its own deterministic training/attack RNG stream.

**Per-client scores for AUC** are defined for every method in `BASELINE_SCORE_DEFINITIONS` (Step 8). Only participating clients have a per-round score; the final client score is the mean of the last `FINAL_K` observed scores — the same rule for all methods. Per-round AUCs are computed over the clients that participated in that round, with honest clients as negatives.


In [ ]:
def method_rng(seed, method):
    """Independent, reproducible training/attack-randomness stream per (seed, method).
    Participation sampling uses its own stream default_rng(seed) shared by all methods, so
    every method sees the SAME client schedule, and the CRTAT branch of run_fl_simulation
    can be reproduced bit-for-bit by run_crtat_variant (Table VII)."""
    return np.random.default_rng([int(seed), METHOD_ORDER.index(method) + 1])

def _client_update(cl, gp, X_c, y_c, cid, rnd, n_rounds, lr, rng,
                   verbose=False, boost=1.0, attackers=True):
    """One client's local round. SHARED by the main simulation, Table VII, the clean-control,
    the influence-bound experiment and Figure 8, so every experiment applies identical
    attack and training code.
      attackers=False → everyone is honest (clean control)
      boost ≠ 1       → every malicious update is multiplied by `boost` (update scaling)
    Attacks: overt = multi-target label flip from round 1; slow = same flip with a linearly
    ramping fraction from SLOW_RAMP_START; inversion = delta × INVERSE_SCALE (= −3: sign flip
    AND ×3 amplification, not a pure −1× flip)."""
    cl.set_params(gp.copy())
    if attackers:
        if cid in POISONED_CLIENTS:
            y_c = inject_multi_label_flip(y_c, fraction=POISON_FRAC, rng=rng)
        elif cid in SLOW_CLIENTS and rnd >= SLOW_RAMP_START:
            frac = min(POISON_FRAC,
                       POISON_FRAC * (rnd - SLOW_RAMP_START + 1) /
                       max(1, n_rounds - SLOW_RAMP_START + 1))
            y_c = inject_multi_label_flip(y_c, fraction=frac, rng=rng)
    cl.train(X_c, y_c, epochs=EPOCHS, lr=lr, rng=rng, verbose=verbose)
    delta = cl.get_params() - gp
    if attackers:
        if cid in INVERSE_CLIENTS:
            delta = INVERSE_SCALE * delta
        if boost != 1.0 and cid in MALICIOUS:
            delta = boost * delta
    return delta

def run_fl_simulation(dataset_name, clients_factory, client_data,
                      X_test, y_test, input_dim,
                      n_rounds=FL_ROUNDS, seed=SEED,
                      client_fraction=CLIENT_FRACTION, early_stop=None,
                      n_classes=N_CLASSES):
    """
    End-to-end FL simulation: every method keeps its OWN global model and its clients train
    on that model each round (it is NOT a replay of fixed recorded updates). Methods are
    matched on seed, data split, client schedule and attack definition; because their global
    models diverge, their updates differ from round to round.

    Returns a 12-tuple (indices 0-10 unchanged from v7/v8):
      0 history, 1 trust_matrix, 2 crtat_matrix, 3 krum_matrix, 4 global_params,
      5 blockchain, 6 det_signals (final per-client score, see final_signal),
      7/8/9 per-round AUC vs honest for overt/slow/inversion (NaN if undefined that round),
      10 inv_matrix, 11 extras (participation masks, per-round scores, timing, Krum clamp log…)
    """
    if early_stop is None: early_stop = USE_EARLY_STOPPING
    rng_sample = np.random.default_rng(seed)
    sample_client = clients_factory()[0]
    init_params = sample_client.get_params().copy()

    CR = 'CRTAT+Blockchain'
    global_params = {m: init_params.copy() for m in METHOD_ORDER}
    blockchain = TrustBlockchain()
    crtat_main = CRTAT()
    tc_base    = TemporalConsistencyBaseline()
    rng_train  = {m: method_rng(seed, m) for m in METHOD_ORDER}

    history = {m: {'acc': []} for m in METHOD_ORDER}
    all_trust, all_crtat, all_krum, all_inv, all_mask = [], [], [], [], []
    round_scores = {m: [] for m in METHOD_ORDER + ['TemporalConsistency']}
    auc_overt   = {m: [] for m in METHOD_ORDER}
    auc_slow    = {m: [] for m in METHOD_ORDER}
    auc_inverse = {m: [] for m in METHOD_ORDER}
    krum_f_used = []
    timing = dict(train_s={m: 0.0 for m in METHOD_ORDER}, detect_s=0.0, aggregate_s=0.0,
                  ledger_build_s=0.0, ledger_verify_s=0.0, total_s=0.0)
    t_run0 = time.perf_counter()

    n_select = max(1, int(round(NUM_CLIENTS * client_fraction)))
    bar = '═'*72
    print(f'\n{bar}')
    print(f'  FL Simulation : {dataset_name}  |  Seed={seed}')
    print(f'  {n_rounds} rounds  |  {NUM_CLIENTS} clients  |  '
          f'{n_select}/{NUM_CLIENTS} selected/round  |  Input dim: {input_dim}  |  Classes: {n_classes}')
    print(f'  Attack frac={POISON_FRAC:.0%}  |  Slow ramp from rnd {SLOW_RAMP_START}  |  '
          f'Inv scale={INVERSE_SCALE}  |  Early-stop={early_stop}')
    print(bar)

    best_avg_acc, rounds_no_improve, stopped_early_at = -1.0, 0, None

    def _obs(sub, participating, fill=np.nan):
        full = np.full(NUM_CLIENTS, fill, dtype=float)
        for k, cid in enumerate(participating): full[cid] = sub[k]
        return full

    last_updates = None
    for rnd in range(1, n_rounds+1):
        lr = cosine_lr(rnd, n_rounds)
        participating = sorted(rng_sample.choice(NUM_CLIENTS, n_select, replace=False).tolist())
        participated  = np.zeros(NUM_CLIENTS, dtype=bool); participated[participating] = True

        method_updates = {}
        for method in METHOD_ORDER:
            t0 = time.perf_counter()
            clients = clients_factory()
            updates = [np.zeros_like(global_params[method]) for _ in range(NUM_CLIENTS)]
            gp = global_params[method]
            for cid in participating:
                X_c, y_c = client_data[cid]
                _verbose = (method == CR and cid == 20 and rnd <= 3)
                if _verbose:
                    print(f'    [Round {rnd} | client C{cid:02d} (honest reference) | samples={len(X_c)}]')
                updates[cid] = _client_update(clients[cid], gp, X_c, y_c, cid, rnd, n_rounds, lr,
                                              rng_train[method], verbose=_verbose)
            method_updates[method] = updates
            timing['train_s'][method] += time.perf_counter() - t0

        upd_part = {m: [method_updates[m][cid] for cid in participating] for m in method_updates}

        # ── CRTAT+Blockchain: detector computation ──
        t0 = time.perf_counter()
        trust_sub  = blockchain_trust(upd_part[CR])
        inv_sub    = inversion_signal(upd_part[CR])
        trust_full = _obs(trust_sub, participating, fill=NEUTRAL_TRUST)   # absent → NEUTRAL_TRUST
        inv_full   = _obs(inv_sub,   participating, fill=0.0)
        crtat_main.update(trust_full, participated_mask=participated, inv_scores=inv_full)
        crtat_sc   = crtat_main.combined_score(trust_full)
        timing['detect_s'] += time.perf_counter() - t0
        last_updates = (list(participating), [u.copy() for u in upd_part[CR]])

        # temporal-consistency baseline (detector only; does not change any aggregation)
        tc_vec = tc_base.update(participating, upd_part[CR]); tc_vec[~participated] = np.nan

        # ── baselines: per-round anomaly scores (observed clients only; NaN elsewhere) ──
        f_used = krum_effective_f(len(participating), NUM_MALICIOUS); krum_f_used.append(f_used)
        krum_w_sub, krum_sc_raw = krum_select(upd_part['Krum'], f=NUM_MALICIOUS)
        krum_sc_sub = (krum_sc_raw - krum_sc_raw.min()) / (np.ptp(krum_sc_raw) + 1e-10)

        tm_agg   = trimmed_mean_agg(upd_part['TrimmedMean'])
        tm_d     = np.array([np.linalg.norm(u - tm_agg) for u in upd_part['TrimmedMean']])
        tm_sub   = (tm_d - tm_d.min()) / (np.ptp(tm_d) + 1e-10)

        fg_w_raw_sub = foolsgold_weights(upd_part['FoolsGold'])
        fg_sub       = 1 - fg_w_raw_sub / (fg_w_raw_sub.max() + 1e-10)

        med_agg_coord = median_agg(upd_part['CoordMedian'])
        med_d    = np.array([np.linalg.norm(u - med_agg_coord) for u in upd_part['CoordMedian']])
        med_sub  = (med_d - med_d.min()) / (np.ptp(med_d) + 1e-10)

        trust_fa_sub = blockchain_trust(upd_part['FedAvg_NoDefense'])
        fa_sub       = 1 - (trust_fa_sub + 1) / 2

        rs_now = {CR: np.where(participated, crtat_sc, np.nan),
                  'Krum': _obs(krum_sc_sub, participating), 'TrimmedMean': _obs(tm_sub, participating),
                  'FoolsGold': _obs(fg_sub, participating), 'CoordMedian': _obs(med_sub, participating),
                  'FedAvg_NoDefense': _obs(fa_sub, participating), 'TemporalConsistency': tc_vec}
        for m, v in rs_now.items(): round_scores[m].append(v)

        all_trust.append(trust_full.copy()); all_crtat.append(crtat_sc.copy())
        all_krum.append(np.nan_to_num(rs_now['Krum'], nan=0.0)); all_inv.append(inv_full.copy())
        all_mask.append(participated.copy())

        for m in METHOD_ORDER:
            auc_overt[m].append(  round_auc(rs_now[m], participated, POISONED_CLIENTS))
            auc_slow[m].append(   round_auc(rs_now[m], participated, SLOW_CLIENTS))
            auc_inverse[m].append(round_auc(rs_now[m], participated, INVERSE_CLIENTS))

        # ── ledger construction ──
        t0 = time.perf_counter()
        blockchain.add_round(rnd, trust_full, crtat_sc, [], FLAG_THRESHOLD)
        timing['ledger_build_s'] += time.perf_counter() - t0

        # ── aggregation ──
        t0 = time.perf_counter()
        crtat_delta = fedavg_agg(upd_part[CR], crtat_weights(crtat_sc[participating]))
        timing['aggregate_s'] += time.perf_counter() - t0
        agg_map = {
            CR                 : crtat_delta,
            'Krum'             : fedavg_agg(upd_part['Krum'], krum_w_sub),
            'TrimmedMean'      : tm_agg,
            'FoolsGold'        : fedavg_agg(upd_part['FoolsGold'], fg_w_raw_sub),
            'CoordMedian'      : med_agg_coord,
            'FedAvg_NoDefense' : fedavg_agg(upd_part['FedAvg_NoDefense']),
        }
        for method, delta in agg_map.items():
            global_params[method] = global_params[method] + delta
            acc, _ = evaluate_model(global_params[method], X_test, y_test,
                                    input_dim=input_dim, n_classes=n_classes)
            history[method]['acc'].append(acc)

        if rnd % 2 == 0 or rnd == 1 or rnd == n_rounds:
            print(f'  Rnd {rnd:>2}/{n_rounds} lr={lr:.4f} | '
                  f'CRTAT+BC:{history[CR]["acc"][-1]*100:5.1f}% '
                  f'Krum:{history["Krum"]["acc"][-1]*100:5.1f}% '
                  f'Trim:{history["TrimmedMean"]["acc"][-1]*100:5.1f}% '
                  f'FG:{history["FoolsGold"]["acc"][-1]*100:5.1f}% '
                  f'Med:{history["CoordMedian"]["acc"][-1]*100:5.1f}% '
                  f'None:{history["FedAvg_NoDefense"]["acc"][-1]*100:5.1f}%')

        if early_stop and rnd >= EARLY_STOP_MIN_ROUND:
            avg_acc = np.mean([history[m]['acc'][-1] for m in history])
            if avg_acc > best_avg_acc + EARLY_STOP_DELTA:
                best_avg_acc, rounds_no_improve = avg_acc, 0
            else:
                rounds_no_improve += 1
                if rounds_no_improve >= EARLY_STOP_PATIENCE:
                    stopped_early_at = rnd
                    print(f'  ⏹  Early stop at round {rnd}.'); break

    print(bar)
    t0 = time.perf_counter(); valid, msg = blockchain.is_valid(); timing['ledger_verify_s'] = time.perf_counter() - t0
    timing['total_s'] = time.perf_counter() - t_run0
    print(f'  Blockchain integrity: {msg}')
    if stopped_early_at is not None: print(f'  Stopped early at round {stopped_early_at}/{n_rounds}.')
    clamped = sum(1 for f in krum_f_used if f < NUM_MALICIOUS)
    print(f'  Krum: requested f={NUM_MALICIOUS}, used f_eff={sorted(set(krum_f_used))} '
          f'(clamped in {clamped}/{len(krum_f_used)} rounds so that n>2f+2 holds)')
    print(bar)

    rs_arr = {m: np.array(v) for m, v in round_scores.items()}
    det_signals = {m: final_signal(rs_arr[m]) for m in rs_arr}          # ONE rule for all methods
    extras = dict(mask=np.array(all_mask), round_scores=rs_arr, krum_f_used=krum_f_used,
                  krum_f_requested=NUM_MALICIOUS, n_participants=n_select, timing=timing,
                  stopped_early_at=stopped_early_at, seed=seed, last_updates=last_updates,
                  early_stop=early_stop)
    return (history, np.array(all_trust), np.array(all_crtat), np.array(all_krum),
            global_params, blockchain, det_signals,
            auc_overt, auc_slow, auc_inverse, np.array(all_inv), extras)

print('✅ run_fl_simulation() defined (v9: shared client-update code, separated RNG streams, '
      'participation-aware scoring, timing + Krum-clamp logging).')


✅ run_fl_simulation() defined (v9: shared client-update code, separated RNG streams, participation-aware scoring, timing + Krum-clamp logging).
